# Objectives

- View the Liquidity provider token as derivative
- Compare that to a standard derivative 
- Understand why an LP might provide liquidity other than to simply earn fees

# Prerequisites

- Metamask
- Web3 python

# Derivatives and their payoff functions

Investing has grown more complicated in recent decades with the creation of numerous derivative instruments offering new ways to manage money. The use of derivatives to hedge risk or improve returns has been around for generations.

**Options** are the simplest derivative investment. Their value is tied to the value of the contract's underlying security. Options give a buyer the opportunity to buy or sell the underlying security. The investor does not own the underlying asset but they make a bet on the direction of its price movement.

The payoff function of an option shows the profit/loss obtained from an option depending on its market price.

There are many types of derivative instruments, including options, swaps, futures, and forward contracts. Derivatives have numerous uses and various levels of risks but are generally considered a sound way to participate in the financial markets.

## Example: Call/Put Options

Here, we only discuss European options.

### Call Option 

A call option is a type of option that gives the holder the right to **buy** the underlying asset at a specified price, known as the strike price, before the option expires, with the cost of the premium. If the price of the underlying asset rises above the strike price, the holder can exercise the option and purchase the asset at the lower strike price, then sell it at the higher market price for a profit. If the price does not rise above the strike price, the holder can choose not to exercise the option and simply let it expire worthless. 

![Call Option Payoff Function](./img/calloption.png)

- $S$ is the price of the underlying asset at expiration
- $X$ is the strike price of the option
- Breakeven point: $S = X + \text{option price}$

### Put Option

On the other hand, a put option is a type of option that gives the holder the right to **sell** the underlying asset at a specified price, before the option expires. If the price of the underlying asset falls below the strike price, the holder can exercise the option and sell the asset at the higher strike price, then buy it back at the lower market price for a profit. If the price does not fall below the strike price, the holder can choose not to exercise the option and simply let it expire worthless. In both cases, the holder pays a premium for the option contract, which is the price of the option, and if the option is not exercised, this premium represents a loss for the holder.

![Put Option Payoff Function](./img/putoption.png)


- Put token USTUSD as 0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c
- Put token MiniDAI as 0xBB6d33A3f5E93DEd0e2F7fd19E77FdDDBd1d0B21
- Put pool address as 0xc9D9C8dD62211D2d4F5E941Fa9A91AF686B290c4

## Connect to web3 API

In [2]:
from web3 import Web3
import json
import os

infura_key = '2e306bdddc7843108fe30334b2dfcfb2'
wallet_public_address = Web3.to_checksum_address('0xECfa7eCDAd56aeb78c4B5319b9446f34D2F68969')
wallet_private_key = '7be192a533a03484ba349c66e3c76566167115c577f282a48524177d88a8a9df'

USTUSD_address = Web3.to_checksum_address('0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c')
web3 = Web3(Web3.HTTPProvider(f'https://sepolia.infura.io/v3/{infura_key}'))
print("Connected to Sepolia Testnet:", web3)

abi_file_path = os.path.join('./abis.json')
try:
    with open(abi_file_path, 'r', encoding='utf-8') as f:
        abi_data = json.load(f)
    print("ABI loaded successfully.")
except Exception as e:
    print(f"Error loading ABI: {e}")


Connected to Sepolia Testnet: <web3.main.Web3 object at 0x00000189013210C0>
ABI loaded successfully.


## Identify the MiniDAI/USTUSD pool

In [22]:
factory_addr = Web3.to_checksum_address('0x0227628f3F023bb0B980b67D528571c95c6DaC1c') #Uniswap V3 factory contract address sepolia
abi_factory = abi_data["abi_factory"]
factory_contract = web3.eth.contract(factory_addr, abi=abi_factory)

dai_token_address = web3.to_checksum_address('0xBB6d33A3f5E93DEd0e2F7fd19E77FdDDBd1d0B21')
USTUSD_token_address = web3.to_checksum_address('0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c')
abi_dai = abi_data["abi_dai"]
abi_USTUSD = abi_data["abi_USTUSD"]
MiniDAI_contract = web3.eth.contract(dai_token_address, abi=abi_dai)
USTUSD_contract = web3.eth.contract(USTUSD_token_address, abi=abi_USTUSD)

def get_pool_address(tokenA, tokenB, fee,factory_contract):
    # Ensure tokens are in correct order (Uniswap V3 requires sorted token addresses)
    if tokenA > tokenB:
        tokenA, tokenB = tokenB, tokenA

    # Call the getPool function
    pool_address = factory_contract.functions.getPool(tokenA, tokenB, fee).call()
    return pool_address

MiniDAI_USTUSD_pool_address = get_pool_address(USTUSD_token_address, dai_token_address, 100, factory_contract)

print("Pool with fee 0.01% :",MiniDAI_USTUSD_pool_address)


Pool with fee 0.01% : 0xc9D9C8dD62211D2d4F5E941Fa9A91AF686B290c4


In [23]:
abi_pool = abi_data["abi_pool"]

MiniDAI_USTUSD_pool_V3 = web3.eth.contract(address=MiniDAI_USTUSD_pool_address, abi=abi_pool)

sqrtPriceX96 = MiniDAI_USTUSD_pool_V3.functions.slot0().call()[0]
raw_price = (sqrtPriceX96 ** 2) / (2 ** 192)

print(f"1 MiniDAI = {raw_price} USTUSD")


1 MiniDAI = 1.0 USTUSD


## Compare Uniswap V2 and V3

Uniswap V2 and V3 has differernt liquidity function, which is generated by the range of pool. In Uniswap V2, we set the range is $\left( 0,\infty \right)$ and in Uniswap V3, we set the price range $\left[ p_a,p_b\right]$ at the beginning, which is can be seen in the following picture. This picture depicts the relationship for a position on a range $\left[ p_a,p_b\right]$ and a current price $p_c \in \left[ p_a,p_b\right]$. $x_{real}$ and $y_{real}$
denote the position’s real reserves. If you are interested in more details, please see [here](https://uniswap.org/whitepaper-v3.pdf).

![Uniswap V3 Liquidity Function](./img/UniswapV3.png)

When calculate the payoff with different price, we first need the value of price, there are two ways to get. Traditionally, we calculate the price of Uniswap V2 and V3 by definition. 

In Uniswap V2, liquidity was distributed uniformly along the x*y=K, then we know that $(x-\Delta x)(y+\Delta y) = x*y$.

Then it's easy to conduct the price in Uniswap V2: $\frac{\Delta y}{\Delta x} = \frac{y}{x}$. 

In Uniswap V3, The amount of liquidity provided can be measured by the value $L$, which is equal to $\sqrt K$. The real reserves of a position are described by the curve: $(x+\frac{L}{\sqrt p_b})(y+L\sqrt p_a)=L^2$. Similarly, we can calculate the price in Uniswap V3: $\frac{\Delta y}{\Delta x} = \frac{y+L\sqrt p_a}{x+\frac{L}{\sqrt p_b}}$. But because of the existence of ``Unclaimed fees`` in the Uniswap pool, this traditional way is not credible.

So, we have to use another way - ``slot0`` function. Please note the positions of tokenA and tokeB in this method. 



## Define the payoff function

In [ ]:
import math

x0 = MiniDAI_contract.functions.balanceOf(MiniDAI_USTUSD_pool_address).call() / 1e18
y0 = USTUSD_contract.functions.balanceOf(MiniDAI_USTUSD_pool_address).call() / 1e18
k = x0 * y0

def getLPpayoff(mode="v3"):
    '''
    p = price of MiniDAI in USTUSD
    payoff = the value of LP token in USTUSD
    '''
    x = MiniDAI_contract.functions.balanceOf(MiniDAI_USTUSD_pool_address).call() / 1e18
    y = USTUSD_contract.functions.balanceOf(MiniDAI_USTUSD_pool_address).call() / 1e18
    p = MiniDAI_USTUSD_pool_V3.functions.slot0().call()[0]**2 / 2**192

    if mode.lower() == "v3": # todo: need to calculate the payoff of V3
        payoff = y + p * x
    elif mode.lower() == "v2": # todo: need to calculate the payoff of V2

    else:
        raise ValueError("mode must be v2 or v3")

    return {'price': p, 'payoff': payoff}

## Calculate LP token payoffs

We now calculate LP payoffs for a range of prices. To do that, the TA will manipualte the pool price by doing large swaps.

**Current Pool V3**
- 500000 MiniDAI
- 500000 USTUSD
- Price: 1 MiniDAI = 1 USTUSD
- min price: 0.8
- max price: 1.4

In [24]:
import json
'''
{
v2: [
    {
        "price": 1.0,
        "payoff": 1000.0
    },
    {
        "price": 1.1,
        "payoff": 900.0
    },
    ...
],
v3: [
    {
        "price": 1.0,
        "payoff": 1000.0
    },
    {
        "price": 1.1,
        "payoff": 900.0
    },
    ...
]
}
'''
def append_to_json(data, v2_or_v3, filename='payoff_data.json'):
    if not os.path.exists(filename):
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump({"v2": [], "v3": []}, f, ensure_ascii=False, indent=4)
    with open(filename, 'r', encoding='utf-8') as f:
        existing_data = json.load(f)
    existing_data[v2_or_v3].append(data)
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(existing_data, f, ensure_ascii=False, indent=4)
def load_data_from_json(filename='payoff_data.json'):
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        data = {"v2": [], "v3": []}
    return data

### Waiting for the TA to manipulate the price...

In [32]:
data = load_data_from_json()
iteration = len(data['v2']) + 1
print(f"Current iteration: {iteration}")

result_v2 = getLPpayoff(mode="V2")
print(f"V2 result: {result_v2}")
append_to_json(result_v2, v2_or_v3="v2")

result_v3 = getLPpayoff(mode="V3")
print(f"V3 result: {result_v3}")
append_to_json(result_v3, v2_or_v3="v3")

Current iteration: 3
V2 result: {'price': 0.8643065660204665, 'payoff': 781642.9004401654}
V3 result: {'price': 0.8643065660204665, 'payoff': 757286.5065752092}


### Waiting for the TA to manipulate the price...

## Plot LP token payoffs

In [ ]:
# draw the plot for V2
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot(MiniDAI_prices_V2,USTUSD_payoffs_V2, marker = 'o')
plt.xlabel('MiniDAI price in USTUSD')
plt.ylabel('LP payoff in USTUSD')
plt.title('Calculate LP token payoff of V2')
plt.grid(True)
plt.show()

In [ ]:
# draw the plot for V3

plt.figure(figsize=(10, 6))
plt.plot(MiniDAI_prices_V3,USTUSD_payoffs_V3, marker = 'o')
plt.xlabel('MiniDAI price in USTUSD')
plt.ylabel('LP payoff in USTUSD')
plt.title('Calculate LP token payoff of V3')
plt.grid(True)
plt.show()

Make sure to save the plot obtained above on your machine.

## Replicating market makers

Turns out you can generate a rich family of payoffs using a CFMM. In fact there is a one-to-one mapping between concave payoffs and CFMM bonding curves! Convex payoffs can be obtained by shorting an LP token.

This implies that LPs might not invest money in pools to just earn fees, they might also want to take bets on prices of the underlying tokens, or hedge their other investments in some way. We can tailor the bonding curve based on the sort of payoff function the LP is looking for.

If anyone is interested to know this mapping and how it is obtained, see [this paper](https://arxiv.org/pdf/2103.14769.pdf). 



# Optional: Perpetuals

**So What is a Perpetual?**

You can think of a Perpetual as a synthetic asset, that tracks the price of an underlying asset. This synthetic asset is created by two sides of the market which are either long - expecting it to go up - or short - expecting it to go down. Within each position, market participants are basically agreeing to buy or sell the asset in the future **at the price of the asset, at the time they acquired the long or short perpetual**. Now if the price goes up, people who agreed to buy it will profit because they have an agreement to buy it for cheaper than the market price, and the same holds vice versa. Oh and one more thing, since you are just agreeing to hypothetically buy or sell the asset in the future, you don't actually have to put up any money to buy it, all you have to put up is some amount of collateral, so that if your position is losing money the protocol has your collateral to pay for your losses, if you lose the value of all of your collateral below some limit defined by the protocol, your position gets liquidated, i.e. the protocol takes your collateral to pay the winners and then sells your position to someone who can put up fresh collateral.

**So how does someone actually profit - or lose money - from a perpetual?**

When they want to close their position, they just sell this long or short perpetual they aquired and will get their profit. If everything is working correctly, they should be able to sell this contract for a profit or loss that corresponds with how they would do if they held the underlying asset. On different protocols "selling" the position can mean selling it on an order book, to an AMM or just getting compensated for their profits to a liquidity pool.  

**How can a protocol ensure that the price of the perpetual will track the underlying asset?**

Afterall, traders may have a general sentiment, for example that the market is pumping and people expect it will go up a lot, people may be willing to pay more for a long perpetual than what the value should be based on the underlying asset price. To make sure that the price doesn't deviate from the underlying asset price, which is necessary to make this whole thing work, traditional perpetual protocols have a funding rate mechanism that charges periodic fees on the overvalued positions and pays those out to the undervalued positiions. This counteracts the market sentiment and drives the price associated with the perpetual to match that of the underlying asset.

This may all be a little confusing because of the terminology used. Here when we say "the price of the perpetual will track the underlying asset" what we really mean is that the profits you will make from selling this contract you have will corespond to the profits you would make if you were just buying and selling x amount of the underlying asset, where x is your leveraged position.

**Recap**

That was a lot. Let's recap this quickly:

*   Perpetuals are synthetic assets that allow you to bet on the future price of an underlying asset.
*   Perpetuals track the value of the underlying asset through a funding rate mechanism that keeps demand of short and long positions balanced.
*   Because perpetuals do not require the purchase or sale of the underlying asset - except for settlement purposes - you can buy a very big position with a small amount of colateral. You just might get wiped out quickly.

Because you are agreeing to buy and sell in the future, perpetuals are considered a futures contract. However, where traditional futures contracts have expiry dates where something has to be exchanged for something, perpetual contracts do not and will last as long as you hold and don't get liquidated.
